# Formal evaluation, temporal validation and gate G4 — NCHS neonatal model

Framework stages: **S8–S9** (formal evaluation and sensitivity analyses) and gate **G4** (framework 1.2). Spanish version: `05_main_evaluation_es.ipynb` (same code). Requires notebook 04 (`data/outputs_04_selection.json`).

- **Models:** the primary learner chosen in notebook 04 with its hyperparameters; logistic regression as reference; and the **reference model** fixed in the outcome card (the primary learner with gestational age and birthweight only), with the paired difference in AUROC.
- **Internal validation:** 3 × 5 stratified cross-validation of the development cohorts (2016–2021), with Platt recalibration inside each training part; bootstrap intervals (200 replicates).
- **Temporal validation (C4):** fit once on 2016–2021, single test on 2022–2023.
- **C1 (framework 1.2):** on the model fitted once on the whole population, its out-of-fold predictions restricted to the newborns alive at the start of day 2 (48-hour landmark, decision M4-D1), with paired bootstrap intervals.
- **Sensitivity analyses of the outcome card:** 28 weeks or more, early neonatal death, cohorts 2020–2021, complete records (decision M4-D3), with and without refit.
- **Thresholds of gate G4 (unchanged):** C1, AUROC drop ≤ 0.05 and at least 2/3 of the AUPRC gain kept; C2, AUROC ≥ 0.75 with lower 95% limit ≥ 0.70; C4, temporal AUROC ≥ 0.70 and calibration slope between 0.90 and 1.10.

Output: `data/outputs_05_main_evaluation.txt`. On a desktop computer it takes in the order of 2 to 4 hours.

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import brentq
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_05_main_evaluation.txt"
SELECTION_FILE = ROOT / "data" / "outputs_04_selection.json"
SEED = 20261123
DEV_YEARS, TEST_YEARS = list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
REPEATS, FOLDS, B = (1, 3, 30) if QUICK else (3, 5, 200)
META = ["unit_id", "group_id", "period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal",
        "ge28", "ga_imputed", "bw_imputed", "n_unknown"]
PREDICTORS = [c for c in pq.read_schema(CONTRACT_DIR / "cohort_2021.parquet").names if c not in META]
REFERENCE = ["gestational_age", "birthweight"]           # reference model fixed in the outcome card
EXTRA = ["period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal", "ge28", "n_unknown"]

# Primary learner and hyperparameters chosen by the preliminary run (notebook 04, selection rule)
sel = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
assert QUICK or not sel["quick"], "notebook 04 was run in QUICK mode: run it again with the full sample"
primary, BEST = sel["primary"], sel["best"]

def make_model(name):
    cfg = dict(BEST[name]); n_iter = cfg.pop("n_estimators", None)
    imp = SimpleImputer(strategy="median")                       # decision M4-D3: no missingness indicators
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=(min(n_iter, 50) if QUICK else n_iter), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, **cfg))

pd.set_option("display.max_colwidth", None)
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)
show("Primary learner (notebook 04)", pd.Series({"primary learner": primary, "hyperparameters": json.dumps(BEST[primary]),
                                                  "ensemble adopted in notebook 04": sel["ensemble_adopted"]}))

## 1. Development and temporal-test samples

In [ ]:
# Samples: every event and FRACTION of non-events, weight 1/FRACTION for non-events (the weighted
# sample stands for the full population). Development 2016-2021; temporal test 2022-2023.
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:     # read in batches and keep the sample only, so the memory stays low
        for b in pq.ParquetFile(CONTRACT_DIR / f"cohort_{y}.parquet").iter_batches(500_000, columns=PREDICTORS + EXTRA):
            c = b.to_pandas()
            n += len(c); e += int(c["label_main"].sum())
            parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
test, n_test, e_test = load(TEST_YEARS, SEED + 1)
show("Data", pd.DataFrame({"births": [n_dev, n_test], "neonatal deaths": [e_dev, e_test],
                           "sample rows": [len(dev), len(test)]}, index=["development 2016-2021", "temporal test 2022-2023"]))

## 2. Internal validation and comparison with the reference model

In [ ]:
# Formal evaluation (framework S8-S9): repeated, recalibrated cross-validation of the primary learner,
# logistic regression and the reference model (the primary learner with gestational age and birthweight only).
# Metrics (weighted where the sampling matters; AUROC does not depend on it)
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
expit = lambda z: 1 / (1 + np.exp(-z))

def cal_slope_intercept(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    f = lambda a: np.sum(w * (y - expit(a + lp)))        # calibration-in-the-large (offset = lp)
    return float(slope), float(brentq(f, -10, 10))

def metrics(y, p, w):
    o = np.argsort(p); cw = np.cumsum(w[o]); m = {}
    for share in (0.05, 0.10):
        cut = p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
        m[f"events in top {int(share * 100)}% (%)"] = 100 * float(w[(p >= cut) & y].sum() / w[y].sum())
    slope, intercept = cal_slope_intercept(y, p, w)
    g = np.empty(len(p), int); g[o] = np.minimum((10 * cw / cw[-1]).astype(int), 9)   # weighted deciles
    ece = sum(abs(np.sum(w[g == k] * y[g == k]) - np.sum(w[g == k] * p[g == k])) for k in range(10)) / w.sum()
    m.update({"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p, sample_weight=w),
              "prevalence": float(np.sum(w * y) / w.sum()), "Brier": float(np.sum(w * (p - y) ** 2) / w.sum()),
              "calibration slope": slope, "calibration intercept": intercept,
              "observed/expected": float(np.sum(w * y) / np.sum(w * p)), "ECE": float(ece)})
    return m

def boot_ci(y, p, w, keys, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y)[0], np.where(~y)[0]; res = []
    for _ in range(B):
        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append({k: v for k, v in metrics(y[ix], p[ix], w[ix]).items() if k in keys})
    r = pd.DataFrame(res)
    return r.quantile(0.025), r.quantile(0.975)

def fit_calibrated(name, X, y, w, seed, cols=None):
    """Fit on 80% of the training part and Platt-recalibrate on the other 20% (inside training)."""
    cols = cols or PREDICTORS
    X = X[cols]
    X_f, X_c, y_f, y_c, w_f, w_c = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=seed)
    m = make_model(name); step = m.steps[-1][0]
    m.fit(X_f, y_f, **{f"{step}__sample_weight": w_f})
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(X_c)[:, 1]).reshape(-1, 1), y_c, sample_weight=w_c)
    return lambda Xn: platt.predict_proba(logit(m.predict_proba(Xn[cols])[:, 1]).reshape(-1, 1))[:, 1]

# Repeated stratified cross-validation (births are independent units; no mother identifier):
# REPEATS x FOLDS, recalibration inside each training part, out-of-fold predictions pooled per repeat.
X_dev, y_dev, w_dev = dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy()
MODELS = {primary: (primary, PREDICTORS), "LR": ("LR", PREDICTORS), "reference (GA + birthweight)": (primary, REFERENCE)}
if primary == "LR":
    MODELS = {"LR": ("LR", PREDICTORS), "reference (GA + birthweight)": ("LR", REFERENCE)}
oof = {name: np.zeros((REPEATS, len(dev))) for name in MODELS}
for r in range(REPEATS):
    t = time.time()
    for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED + r).split(X_dev, y_dev):
        for name, (learner, cols) in MODELS.items():
            f = fit_calibrated(learner, X_dev.iloc[tr], y_dev[tr], w_dev[tr], SEED + r, cols)
            oof[name][r, te] = f(X_dev.iloc[te])
    print(f"repeat {r + 1} done in {(time.time() - t) / 60:.1f} min")

KEYS = ["AUROC", "AUPRC", "calibration slope", "calibration intercept", "observed/expected"]
rows = {}
for name in oof:
    per_rep = pd.DataFrame([metrics(y_dev, oof[name][r], w_dev) for r in range(REPEATS)])
    lo, hi = boot_ci(y_dev, oof[name][0], w_dev, KEYS, SEED)
    row = per_rep.mean().round(4).to_dict()
    row.update({f"{k} 2.5%": round(lo[k], 4) for k in KEYS}); row.update({f"{k} 97.5%": round(hi[k], 4) for k in KEYS})
    row["AUROC SD across repeats"] = round(per_rep["AUROC"].std(ddof=0), 4)
    rows[name] = row
main = pd.DataFrame(rows)
show(f"Internal validation: {REPEATS} x {FOLDS}-fold cross-validation, recalibrated (development 2016-2021)", main)

# What the rest of the information adds to gestational age and birthweight (paired bootstrap, repeat 1)
rng = np.random.default_rng(SEED + 5); pos, neg = np.where(y_dev)[0], np.where(~y_dev)[0]
REF = "reference (GA + birthweight)"
d_auc = []
for _ in range(B):
    ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
    d_auc.append(roc_auc_score(y_dev[ix], oof[primary][0][ix]) - roc_auc_score(y_dev[ix], oof[REF][0][ix]))
show("Full model minus reference model (GA + birthweight): AUROC difference, paired bootstrap 95% CI",
     f"{roc_auc_score(y_dev, oof[primary][0]) - roc_auc_score(y_dev, oof[REF][0]):+.4f} "
     f"({np.percentile(d_auc, 2.5):+.4f} to {np.percentile(d_auc, 97.5):+.4f})")

## 3. Temporal validation, criterion C1 and sensitivity analyses

In [ ]:
# Temporal validation (criterion C4): fit once on the whole development sample (2016-2021, which passes
# gate G3, notebook 03), recalibrate inside it, apply once to 2022-2023.
X_t, y_t, w_t = test[PREDICTORS], test["label_main"].to_numpy(bool), test["w"].to_numpy()
temporal, p_temporal = {}, {}
for name, (learner, cols) in MODELS.items():
    f = fit_calibrated(learner, X_dev, y_dev, w_dev, SEED, cols)
    p_t = p_temporal[name] = f(X_t)
    m = metrics(y_t, p_t, w_t); lo, hi = boot_ci(y_t, p_t, w_t, KEYS, SEED + 7)
    temporal[name] = {**{k: round(v, 4) for k, v in m.items()},
                      **{f"{k} 2.5%": round(lo[k], 4) for k in KEYS}, **{f"{k} 97.5%": round(hi[k], 4) for k in KEYS}}
show("Temporal validation: development 2016-2021, single test 2022-2023", pd.DataFrame(temporal))

# Criterion C1 (framework 1.2): evaluated on the model fitted once on the whole population. Its out-of-fold
# predictions are restricted to the newborns alive at the start of day 2 (48-hour landmark, decision M4-D1),
# where the event is death on days 2-27. Paired bootstrap of the full population and the landmark subset.
p0 = oof[primary][0]
lm = dev["at_risk_48h"].to_numpy(bool); y_lm = dev["label_day2_27"].to_numpy(bool)
def c1_quantities(ix):
    yf, pf, wf = y_dev[ix], p0[ix], w_dev[ix]
    sub = lm[ix]; ys, ps, ws = y_lm[ix][sub], pf[sub], wf[sub]
    a_f, a_s = roc_auc_score(yf, pf), roc_auc_score(ys, ps)
    prev_f, prev_s = np.sum(wf * yf) / wf.sum(), np.sum(ws * ys) / ws.sum()
    g_f = average_precision_score(yf, pf, sample_weight=wf) - prev_f
    g_s = average_precision_score(ys, ps, sample_weight=ws) - prev_s
    return {"AUROC full": a_f, "AUROC landmark": a_s, "AUROC drop": a_f - a_s,
            "AUPRC gain full": g_f, "AUPRC gain landmark": g_s, "AUPRC gain kept (%)": 100 * g_s / g_f}
point = c1_quantities(np.arange(len(dev)))
rng = np.random.default_rng(SEED + 11); pos, neg = np.where(y_dev)[0], np.where(~y_dev)[0]
bs = pd.DataFrame([c1_quantities(np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])) for _ in range(B)])
c1_tab = pd.DataFrame({"estimate": pd.Series(point), "2.5%": bs.quantile(0.025), "97.5%": bs.quantile(0.975)}).round(4)
show(f"Criterion C1: 48-hour landmark on the out-of-fold predictions of the primary learner "
     f"({int(y_lm[lm].sum())} deaths on days 2-27; {int((y_dev & ~lm).sum())} deaths on days 0-1 removed)", c1_tab)

# Sensitivity analyses of the outcome card (S9) on the same out-of-fold predictions (no refit)
ge28 = dev["ge28"].to_numpy(bool); early = dev["label_early_neonatal"].to_numpy(bool)
late = y_dev & ~early; pandemic = dev["period"].isin([2020, 2021]).to_numpy()
complete = (dev["n_unknown"] == 0).to_numpy()
sens = {"all births (main)": (np.ones(len(dev), bool), y_dev),
        "(c) 28 weeks or more": (ge28, y_dev),
        "(a) + (c) 48-hour landmark, 28 weeks or more": (lm & ge28, y_lm),
        "(b) early neonatal death 0-6 days vs survivors (deaths on days 7-27 removed)": (~late, early),
        "cohorts 2020-2021 (pandemic)": (pandemic, y_dev),
        "cohorts 2016-2019": (~pandemic, y_dev),
        "complete records (main model, no refit)": (complete, y_dev)}
srows = {}
for k, (mask, yy) in sens.items():
    ww, pp = w_dev[mask], p0[mask]
    srows[k] = {"births (weighted)": int(round(ww.sum())), "events": int(yy[mask].sum()),
                "AUROC": round(roc_auc_score(yy[mask], pp), 4),
                "observed/expected": round(float(np.sum(ww * yy[mask]) / np.sum(ww * pp)), 3)}
show("Sensitivity analyses (primary learner, out-of-fold predictions of the model fitted once; observed/expected "
     "compares the deaths of each label with the predicted risk of death at 0-27 days)", pd.DataFrame(srows).T)

# Refitted sensitivities (each cross-validated with the primary learner and its tuned hyperparameters).
# Complete records (decision M4-D3): the set has tens of thousands of events, far above the 104 required by
# gate G3 for 71 parameters (notebook 03).
def cv_auroc(mask, cols, label):
    Xs, ys, ws = dev.loc[mask, cols], dev.loc[mask, label].to_numpy(bool), w_dev[mask]
    pp = np.zeros(len(Xs))
    for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(Xs, ys):
        pp[te] = fit_calibrated(primary, Xs.iloc[tr], ys[tr], ws[tr], SEED, cols)(Xs.iloc[te])
    return round(roc_auc_score(ys, pp), 4), int(ys.sum())
rrows = {"complete records, refitted (no imputation)": cv_auroc(complete, PREDICTORS, "label_main"),
         "48-hour landmark, refitted": cv_auroc(lm, PREDICTORS, "label_day2_27"),
         "28 weeks or more, refitted": cv_auroc(ge28, PREDICTORS, "label_main"),
         "28 weeks or more, reference model (GA + birthweight), refitted": cv_auroc(ge28, REFERENCE, "label_main")}
show("Refitted sensitivity analyses (primary learner): AUROC and events", pd.DataFrame(rrows, index=["AUROC", "events"]).T)

## 4. Gate G4, confusion matrix and decision curve

In [ ]:
# Gate G4 (classification, framework 1.2).
# C1: 48-hour landmark on the model fitted once: AUROC drop <= 0.05 and >= 2/3 of the AUPRC gain kept.
# C2: every death occurs after the prediction point (birth + 5 minutes): AUROC >= 0.75 with lower 95% limit >= 0.70.
# C3: not applicable (the outcome comes from the death certificate, not from a test).
# C4: temporal AUROC >= 0.70 and temporal calibration slope between 0.90 and 1.10.
x, tt = main[primary], temporal[primary]
c1 = (point["AUROC drop"] <= 0.05) and (point["AUPRC gain kept (%)"] >= 100 * 2 / 3)
c2 = (x["AUROC"] >= 0.75) and (x["AUROC 2.5%"] >= 0.70)
c4 = (tt["AUROC"] >= 0.70) and (0.90 <= tt["calibration slope"] <= 1.10)
top10 = x["events in top 10% (%)"] / 10
if x["AUROC"] >= 0.75 and c1 and c2 and c4:
    cls = "PREDICTOR"
elif x["AUROC"] >= 0.75 and top10 >= 2:
    cls = "EARLY IDENTIFIER"
else:
    cls = "NEITHER"
if tt["AUROC"] < 0.65:
    cls = "NEITHER"
show("Gate G4: classification of the neonatal model (primary learner)", pd.Series({
    "C1 events already present": f"{'met' if c1 else 'NOT met'} (48-hour landmark: AUROC drop {point['AUROC drop']:.3f}, "
        f"95% CI {c1_tab.loc['AUROC drop', '2.5%']:.3f}-{c1_tab.loc['AUROC drop', '97.5%']:.3f}, required <= 0.05; "
        f"AUPRC gain kept {point['AUPRC gain kept (%)']:.0f}%, 95% CI {c1_tab.loc['AUPRC gain kept (%)', '2.5%']:.0f}-"
        f"{c1_tab.loc['AUPRC gain kept (%)', '97.5%']:.0f}%, required >= 67%)",
    "C2 outcome with time": f"{'met' if c2 else 'NOT met'} (AUROC {x['AUROC']:.3f}, 95% CI {x['AUROC 2.5%']:.3f}-{x['AUROC 97.5%']:.3f}; required >= 0.75 and >= 0.70)",
    "C3 detection bias": "not applicable (death certificate; linkage of deaths to births 100%, notebook 01)",
    "C4 stability over time": f"{'met' if c4 else 'NOT met'} (temporal AUROC {tt['AUROC']:.3f}, 95% CI {tt['AUROC 2.5%']:.3f}-{tt['AUROC 97.5%']:.3f}; "
        f"slope {tt['calibration slope']:.2f}, 95% CI {tt['calibration slope 2.5%']:.2f}-{tt['calibration slope 97.5%']:.2f}, required 0.90-1.10)",
    "events in top 10% vs chance": f"{top10:.2f} times",
    "classification": cls}))
# Accuracy and confusion matrix of the primary model (requested; not a framework metric)
cm = []
for share in (0.10, 0.05):
    p = oof[primary][0]; o = np.argsort(p); cw = np.cumsum(w_dev[o])
    flag = p >= p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
    tp, fn = w_dev[flag & y_dev].sum(), w_dev[~flag & y_dev].sum()
    fp, tn = w_dev[flag & ~y_dev].sum(), w_dev[~flag & ~y_dev].sum()
    cm.append({"flagged": f"top {int(share * 100)}%", "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
               "accuracy": round((tp + tn) / (tp + tn + fp + fn), 4), "sensitivity": round(tp / (tp + fn), 3),
               "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4), "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrix of the primary learner (out-of-fold, weighted to the development population)", pd.DataFrame(cm).set_index("flagged"))

# Decision curve (net benefit) at a few risk thresholds
dc = []
for thr in (0.002, 0.005, 0.01, 0.02, 0.05):
    p = oof[primary][0]; flag = p >= thr; n = w_dev.sum()
    nb = w_dev[flag & y_dev].sum() / n - w_dev[flag & ~y_dev].sum() / n * thr / (1 - thr)
    prev = w_dev[y_dev].sum() / n
    dc.append({"threshold": thr, "net benefit model": round(nb, 5), "net benefit treat all": round(prev - (1 - prev) * thr / (1 - thr), 5),
               "% flagged": round(100 * w_dev[flag].sum() / n, 2)})
show("Decision curve (primary learner)", pd.DataFrame(dc).set_index("threshold"))

OUT_FILE.write_text("Main evaluation of the neonatal model (birth + 5 minutes)\n" + "\n".join(lines) + "\n", encoding="utf-8")